# AEZ-26 Phase 2F — Final Naming/QC Patch

This is a **rename/QC-only patch** for the already computed Phase 2F outputs.

It does **not** recalculate:
- seasonal anomalies,
- response depths,
- lags,
- event labels,
- recovery dates.

It only corrects the interpretation of signed recovery offsets relative to the flash-drought event end.

The patched outputs are saved to `Phase2/Phase2F_FINAL/` and packaged as one ZIP.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import pandas as pd
import shutil

BASE_DIR = Path('/content/drive/MyDrive/AEZ26_Final_Research_Data')
PHASE2_DIR = BASE_DIR / 'Phase2'
SRC_DIR = PHASE2_DIR / 'Phase2F'
OUT_DIR = PHASE2_DIR / 'Phase2F_FINAL'
FIG_OUT = OUT_DIR / 'Figures'

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_OUT.mkdir(parents=True, exist_ok=True)

required = [
    'AEZ26_Phase2F_MOD16_Event_Response_Metrics.csv',
    'AEZ26_Phase2F_MOD13_Event_Response_Metrics.csv',
    'AEZ26_Phase2F_GRACE_TWSA_Event_Response_Metrics.csv',
    'AEZ26_Phase2F_JRC_SurfaceWater_Event_Response_Metrics.csv',
    'AEZ26_Phase2F_Cross_Response_Event_Synthesis.csv',
    'AEZ26_Phase2F_QC.csv',
    'AEZ26_Phase2F_Method_Config.txt'
]

for name in required:
    p = SRC_DIR / name
    print(name, '->', p.exists())
    if not p.exists():
        raise FileNotFoundError(p)


In [ ]:
datasets = {
    'MOD16': 'AEZ26_Phase2F_MOD16_Event_Response_Metrics.csv',
    'MOD13': 'AEZ26_Phase2F_MOD13_Event_Response_Metrics.csv',
    'GRACE': 'AEZ26_Phase2F_GRACE_TWSA_Event_Response_Metrics.csv',
    'JRC': 'AEZ26_Phase2F_JRC_SurfaceWater_Event_Response_Metrics.csv'
}

patched = {}

for prefix, filename in datasets.items():
    df = pd.read_csv(SRC_DIR / filename)

    old_normal = f'{prefix}_recovery_to_seasonal_normal_days_after_event_end'
    old_base = f'{prefix}_recovery_to_prebaseline_days_after_event_end'

    new_normal = f'{prefix}_relative_days_from_event_end_to_seasonal_normal_recovery'
    new_base = f'{prefix}_relative_days_from_event_end_to_prebaseline_recovery'

    df = df.rename(columns={
        old_normal: new_normal,
        old_base: new_base
    })

    df[f'{prefix}_seasonal_normal_recovery_before_event_end_flag'] = (
        pd.to_numeric(df[new_normal], errors='coerce') < 0
    ).astype(int)

    df[f'{prefix}_prebaseline_recovery_before_event_end_flag'] = (
        pd.to_numeric(df[new_base], errors='coerce') < 0
    ).astype(int)

    assert not df.columns.duplicated().any()

    patched[prefix] = df
    df.to_csv(OUT_DIR / filename, index=False)

    print(prefix,
          'seasonal-normal recovery before event end =',
          int(df[f'{prefix}_seasonal_normal_recovery_before_event_end_flag'].sum()))


In [ ]:
# Patch cross-response synthesis consistently.
synthesis_name = 'AEZ26_Phase2F_Cross_Response_Event_Synthesis.csv'
synthesis = pd.read_csv(SRC_DIR / synthesis_name)

for prefix in ['MOD16','MOD13','GRACE','JRC']:
    old_normal = f'{prefix}_recovery_to_seasonal_normal_days_after_event_end'
    old_base = f'{prefix}_recovery_to_prebaseline_days_after_event_end'

    new_normal = f'{prefix}_relative_days_from_event_end_to_seasonal_normal_recovery'
    new_base = f'{prefix}_relative_days_from_event_end_to_prebaseline_recovery'

    synthesis = synthesis.rename(columns={
        old_normal: new_normal,
        old_base: new_base
    })

    synthesis[f'{prefix}_seasonal_normal_recovery_before_event_end_flag'] = (
        pd.to_numeric(synthesis[new_normal], errors='coerce') < 0
    ).astype(int)

    synthesis[f'{prefix}_prebaseline_recovery_before_event_end_flag'] = (
        pd.to_numeric(synthesis[new_base], errors='coerce') < 0
    ).astype(int)

assert len(synthesis) == 39
assert synthesis['event_id'].is_unique
assert not synthesis.columns.duplicated().any()

synthesis.to_csv(OUT_DIR / synthesis_name, index=False)
print('Cross-response synthesis patched: PASS')


In [ ]:
# Copy unchanged Phase 2F tables.
unchanged = [
    'AEZ26_Phase2F_MOD16_Seasonal_Anomaly_Links.csv',
    'AEZ26_Phase2F_MOD13_Seasonal_Anomaly_Links.csv',
    'AEZ26_Phase2F_GRACE_TWSA_Seasonal_Anomaly_Links.csv',
    'AEZ26_Phase2F_JRC_Qualified_Seasonal_Anomaly_Links.csv',
    'AEZ26_Phase2F_JRC_Qualified_Monthly_Climatology.csv'
]

for name in unchanged:
    shutil.copy2(SRC_DIR / name, OUT_DIR / name)

# Copy figures unchanged.
src_fig = SRC_DIR / 'Figures'
if src_fig.exists():
    for p in src_fig.iterdir():
        if p.is_file():
            shutil.copy2(p, FIG_OUT / p.name)

print('Unchanged outputs copied.')


In [ ]:
# Updated QC.
old_qc = pd.read_csv(SRC_DIR / 'AEZ26_Phase2F_QC.csv')

extra = pd.DataFrame([
    {
        'check':'MOD16 seasonal-normal recovery before event end',
        'value':int(patched['MOD16']['MOD16_seasonal_normal_recovery_before_event_end_flag'].sum()),
        'status':'RESULT',
        'note':'Signed timing; not a calculation error.'
    },
    {
        'check':'MOD13 seasonal-normal recovery before event end',
        'value':int(patched['MOD13']['MOD13_seasonal_normal_recovery_before_event_end_flag'].sum()),
        'status':'RESULT',
        'note':'Signed timing; not a calculation error.'
    },
    {
        'check':'GRACE seasonal-normal recovery before event end',
        'value':int(patched['GRACE']['GRACE_seasonal_normal_recovery_before_event_end_flag'].sum()),
        'status':'RESULT',
        'note':'Signed timing; not a calculation error.'
    },
    {
        'check':'JRC seasonal-normal recovery before event end',
        'value':int(patched['JRC']['JRC_seasonal_normal_recovery_before_event_end_flag'].sum()),
        'status':'RESULT',
        'note':'Signed timing; not a calculation error.'
    },
    {
        'check':'Duplicate synthesis columns after patch',
        'value':int(synthesis.columns.duplicated().sum()),
        'status':'PASS',
        'note':'Must be zero.'
    }
])

qc = pd.concat([old_qc, extra], ignore_index=True)
qc.to_csv(OUT_DIR / 'AEZ26_Phase2F_QC.csv', index=False)

method_config = (SRC_DIR / 'AEZ26_Phase2F_Method_Config.txt').read_text(encoding='utf-8')

method_config += '''

FINAL NOMENCLATURE CLARIFICATION
- Recovery is searched after the response minimum; it is not forced to occur after soil-moisture event termination.
- Recovery timing relative to event end is therefore a SIGNED offset.
- Negative value = response recovery occurred before formal flash-drought event end.
- Output columns use `relative_days_from_event_end_to_*_recovery`.
- Explicit `*_recovery_before_event_end_flag` fields are retained.
'''

(OUT_DIR / 'AEZ26_Phase2F_Method_Config.txt').write_text(method_config, encoding='utf-8')

print('Final QC/config saved.')


In [ ]:
ZIP_BASE = PHASE2_DIR / 'AEZ26_Phase2F_FINAL_Output_Package'

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    'zip',
    root_dir=PHASE2_DIR,
    base_dir='Phase2F_FINAL'
)

print('FINAL PATCH ZIP:')
print(zip_path)


## Final check

This patch changes **names and interpretation flags only**. Numerical response values and recovery dates remain unchanged.

Upload the final ZIP if you want a final locked Phase 2F audit package.
